# Visualizing Planetary Computer data with Lonboard

This notebook walks through interactive geospatial visualization with [Lonboard](https://developmentseed.org/lonboard/). Lonboard is a Python library that renders Cloud Optimized GeoTIFFs and vector data on a GPU-accelerated WebGL map directly in Jupyter. Key benefits:

1. **GPU rendering** — handles millions of features without breaking interactivity.
2. **No tile server** — COG tiles stream straight to the browser; no intermediate service.
3. **STAC-native** — `RasterLayer.from_stac()` takes signed PC items directly.
4. **Composable** — stack raster, vector, and overlay layers in one `Map`.
5. **Live mutation** — change opacity, colormap, or data and the map updates in place.

Each cell below renders a map you can pan and zoom.

The companion [Lonboard tutorial](../overview/lonboard.md) has the full narrative.

## Install

In [ ]:
%pip install --quiet lonboard pystac-client planetary-computer geopandas

## Open the Planetary Computer STAC catalog

`modifier=planetary_computer.sign_inplace` signs every asset href as the search returns, so Lonboard can fetch tiles directly without a separate signing step.

**Expected result:** working `catalog` client, no output printed.

In [ ]:
import pystac_client
import planetary_computer

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

## Search NAIP over Portland

**Expected result:** a small handful of items covering the bbox in 2022.

In [ ]:
items = catalog.search(
    collections=["naip"],
    bbox=[-122.7, 45.5, -122.6, 45.6],
    datetime="2022-01-01/2023-01-01",
).item_collection()

len(items)

## Render the imagery

`RasterLayer.from_stac()` takes the list of items and figures out which COG asset to render. The map below is fully interactive — pan and zoom to load tiles on demand.

**Expected result:** an interactive map of Portland with NAIP imagery.

In [ ]:
from lonboard import Map
from lonboard.experimental import RasterLayer

layer = RasterLayer.from_stac(items)
Map(layer)

## Adjust opacity

Mutating the layer in place updates the existing map without re-fetching tiles.

**Expected result:** the same map redraws at 70% opacity.

In [ ]:
layer.opacity = 0.7

## Apply a colormap (single-band rasters)

Useful for NDVI, classification, or any thematic raster. Re-running this cell with a different `colormap_name` (`viridis`, `magma`, `RdBu`, …) compares options without re-fetching tiles.

**Expected result:** a separate map with a styled colormap applied.

In [ ]:
colormapped = RasterLayer.from_stac(items, colormap_name="viridis", rescale=(-1, 1))
Map(colormapped)

## Combine raster with vector overlays

Layers compose. Draw the STAC item footprints over the imagery:

**Expected result:** the NAIP map with item-boundary lines drawn on top.

In [ ]:
import geopandas as gpd
from lonboard import PathLayer

footprints = gpd.GeoDataFrame.from_features(items.to_dict())
footprint_layer = PathLayer.from_geopandas(
    footprints.boundary.to_frame("geometry")
)

Map([layer, footprint_layer])

## You're done

If every cell above rendered a map, the stack is wired up end-to-end: STAC search → signed assets → GPU-rendered raster → vector overlays. Swap in your own bbox, collection, or GeoDataFrame and the same pattern applies.

For pixel-level analysis (window reads, overview traversal), see the [async-geotiff tutorial](../overview/async-geotiff.md). For a standalone web app rather than a notebook, the [deck.gl-raster tutorial](../overview/deckgl-raster.md) builds the same renderer in TypeScript.